# Revisão de Condicionantes

Pipeline para aplicar dicionário curado nos condicionantes.

In [ ]:
import pandas as pd
from pathlib import Path

def get_project_root():
    path = Path.cwd()
    while path != path.parent:
        if (path / "data").exists():
            return path
        path = path.parent
    raise Exception("Raiz do projeto não encontrada")

ROOT = get_project_root()
DIR_REVIEW = ROOT / "data" / "04_review"
DIR_DICIONARIOS = DIR_REVIEW / "dicionarios_manuais"
DIR_SAIDA = DIR_REVIEW / "revisoes_aplicadas"
DIR_LOGS = DIR_REVIEW / "logs_revisao"

for p in [DIR_DICIONARIOS, DIR_SAIDA, DIR_LOGS]: p.mkdir(parents=True, exist_ok=True)

ARQ_ENTRADA = ROOT / "data" / "03_transformed" / "valores_unicos_para_normalizacao" / "valores_unicos_condicionantes.csv"
ARQ_DICIONARIO = DIR_DICIONARIOS / "dicionario_condicionantes.csv"
ARQ_DIM_CURADA = DIR_SAIDA / "dim_condicionante_curada.csv"
ARQ_PENDENCIAS = DIR_SAIDA / "pendencias_condicionantes.csv"

df = pd.read_csv(ARQ_ENTRADA, encoding='utf-8')
print(f"Linhas: {len(df)}")

In [ ]:
if ARQ_DICIONARIO.exists():
    base_revisao = pd.read_csv(ARQ_DICIONARIO, encoding="utf-8-sig")
else:
    base_revisao = df.copy()
    if 'valor_normalizado' not in base_revisao.columns:
        base_revisao['valor_normalizado'] = base_revisao['valor_original']
    
    # Remove missing/empty
    base_revisao['valor_normalizado'] = base_revisao['valor_normalizado'].fillna('').astype(str).str.strip()
    base_revisao = base_revisao[base_revisao['valor_normalizado'] != '']
    
    base_revisao = base_revisao.drop_duplicates('valor_normalizado').copy()
    
    base_revisao['macrocondicionante'] = ""
    base_revisao['manter'] = True
    base_revisao['observacao'] = ""
    
    if 'qtd_ocorrencias' not in base_revisao.columns:
        base_revisao['qtd_ocorrencias'] = 1
    
    base_revisao = base_revisao[['valor_normalizado', 'macrocondicionante', 'manter', 'observacao', 'qtd_ocorrencias']]

print(base_revisao.head())

In [ ]:
regras = [
    {
        "padrao": r"polític|govern|estado|institu|lei|regulament",
        "macrocondicionante": "Político e Institucional",
    },
    {
        "padrao": r"econom|financ|mercad|preço|investimento|renda|custo",
        "macrocondicionante": "Econômico",
    },
    {
        "padrao": r"socia|demográfic|população|cultur|educação|saúde",
        "macrocondicionante": "Social e Demográfico",
    },
    {
        "padrao": r"tecnolog|inovação|digital|ia|automação",
        "macrocondicionante": "Tecnológico",
    },
    {
        "padrao": r"ambient|clima|carbono|energia|recursos|desmatamento|água",
        "macrocondicionante": "Ambiental",
    }
]

def aplicar_regras(base, regras):
    base = base.copy()
    for regra in regras:
        pendente = base["macrocondicionante"].isna() | (base["macrocondicionante"].astype(str).str.strip() == "")
        filtro = base["valor_normalizado"].str.contains(regra["padrao"], case=False, na=False, regex=True) & pendente
        base.loc[filtro, "macrocondicionante"] = regra["macrocondicionante"]
    return base

base_revisao = aplicar_regras(base_revisao, regras)
print("Regras aplicadas.")
print(base_revisao.head())

In [ ]:
pendencias = base_revisao[base_revisao["macrocondicionante"].isna() | (base_revisao["macrocondicionante"].astype(str).str.strip() == "")].copy()

dim_curada = base_revisao[['valor_normalizado', 'macrocondicionante', 'manter', 'observacao']].copy()

base_revisao.to_csv(ARQ_DICIONARIO, index=False, encoding="utf-8-sig")
dim_curada.to_csv(ARQ_DIM_CURADA, index=False, encoding="utf-8-sig")
pendencias.to_csv(ARQ_PENDENCIAS, index=False, encoding="utf-8-sig")

print(f"Total: {len(base_revisao)}, Pendências: {len(pendencias)}")
